# Notebook to compute the stress loss from saved node embeddings from link prediction (LP) or node classification (NC) on different datasets.
### Example: Disease

In [1]:
import numpy as np
import networkx as nx
import pandas as pd
import os
import pickle as pkl
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
from manifolds import Lorentz, Euclidean, PoincareBall

In [2]:
# set paths
root_path = "../"
data_path = root_path + "data/"
embeddings_path = root_path + "data/embeddings/"

In [3]:
# === Model ===
class DistanceScaler(nn.Module):
    def __init__(self, manifold):
        super().__init__()
        self.a = nn.Parameter(torch.tensor(0.0))
        self.b = nn.Parameter(torch.tensor(1.0))
        self.manifold = manifold

    def decode(self, h, idx):
        h_src = h[idx[0]]
        h_dst = h[idx[1]]
        dist = self.manifold.dist(h_src, h_dst)
        return self.a + self.b * dist

class StressLoss(nn.Module):
    """
    Stress loss for pairwise distances (1D tensors).
    """

    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, D_pred: torch.Tensor, D_true: torch.Tensor) -> torch.Tensor:
        # D_pred, D_true: shape [num_pairs]
        weights = 1.0 / (D_true ** 2 + self.eps)
        loss = (weights * (D_pred - D_true) ** 2).mean()
        return loss

In [4]:
# specify task and dataset
task_name = "lp"
dataset_name = "disease"
model_folder_name = ".yaml/"

In [5]:
# # Load edge list
if dataset_name == "disease":
    if task_name == "nc":
        dataset_name="disease_nc"
        edges_path = data_path + "/disease_nc/disease_nc.edges.csv"
        edges_df = pd.read_csv(edges_path, header=None)
        G = nx.from_pandas_edgelist(edges_df, source=edges_df.columns[0], target=edges_df.columns[1])
        feats_path = data_path + "/disease_nc/disease_nc.feats.npz"
        feats_sp = sp.load_npz(feats_path)
        features = np.array(feats_sp.todense())
    elif task_name == "lp":
        dataset_name="disease_lp"
        edges_path = data_path + "/disease_lp/disease_lp.edges.csv"
        edges_df = pd.read_csv(edges_path, header=None)
        G = nx.from_pandas_edgelist(edges_df, source=edges_df.columns[0], target=edges_df.columns[1])
        feats_path = data_path + "/disease_lp/disease_lp.feats.npz"
        feats_sp = sp.load_npz(feats_path)
        features = np.array(feats_sp.todense())
    else:
        raise ValueError(f"Unknown task: {task_name}")
elif dataset_name == "airport":
    path = os.path.join(root_path, 'data/airport/airport.p')
    with open(path, 'rb') as f:
        G = pkl.load(f)
    #sorted_nodes = sorted(G.nodes())
    mapping = {old: i for i, old in enumerate(G.nodes())}
    G = nx.relabel_nodes(G, mapping)
    nodes = list(G.nodes())
    features = np.array([G.nodes[u]['feat'] for u in nodes])
elif dataset_name in ["Cora", "Pubmed", "Citeseer"]:
    path = os.path.join(data_path, dataset_name.lower(), dataset_name, "geom-gcn", "processed", "data.pt")
    with open(path, 'rb') as f:
        data = torch.load(f)
    edges = data[0]['edge_index']
    edges = edges.numpy()
    G = nx.from_edgelist(edges.transpose())
    features = np.array(data[0]['x'])
    num_nodes = features.shape[0]
    G.add_nodes_from(range(num_nodes))
else:
    raise NotImplementedError

# Build graph
print("Is tree?",nx.is_tree(G))  # True
print("Is connected?", nx.is_connected(G))  # True
print("Number of connected components:", nx.number_connected_components(G))
lengths = dict(nx.single_source_shortest_path_length(G, 0))
print("Shortest path lengths from root:", min(lengths))
print("All nodes reached?", set(lengths.keys()) == set(G.nodes()))
print("Maximum distance from root:", max(lengths.values()))

Is tree? True
Is connected? True
Number of connected components: 1
Shortest path lengths from root: 0
All nodes reached? True
Maximum distance from root: 4


In [6]:
# create distance matrix
nodes = list(G.nodes())
n = len(nodes)
dist_matrix = np.full((n, n), np.inf)
lengths = dict(nx.all_pairs_dijkstra_path_length(G))  # Usa Dijkstra,
components = list(nx.connected_components(G))
largest_cc = max(components, key=len)
nodes_indices_lcc=np.array(list(largest_cc))

# Build the matrix
node_index = {node: idx for idx, node in enumerate(nodes)}
for i, u in enumerate(nodes):
    for v, d in lengths[u].items():
        j = node_index[v]
        dist_matrix[i, j] = d

In [7]:
# build training data
dist_matrix_tensor = torch.tensor(dist_matrix).float()  # shortest paths from NetworkX
idx = torch.tensor(np.array([[i, j] for i in range(n) for j in range(i+1, n)]))  # shape [2, num_pairs]
targets = dist_matrix_tensor[idx[:,0], idx[:,1]]                      # shape [num_pairs]
mask = torch.isfinite(targets)
idx = idx[mask]
targets = targets[mask]

# Optional subsampling (usually unnecessary, but can speed up computation for large datasets)
num_samples = 20_000_000
if idx.size(0) > num_samples:
    perm = torch.randperm(idx.size(0))
    idx = idx[perm[:num_samples]]
    targets = targets[perm[:num_samples]]

print(f"Number of distance samples used for stress loss: {idx.size(0)}")

Number of distance samples used for stress loss: 3549780


In [10]:
seeds = [42, 123, 99, 18, 456, 10, 1, 44, 666, 11]
layer = [-1, -1, -1, -1, -1, -1] # default is last layer

embeddings_hybo_L_list = []
embeddings_hybo_E_list = []
embeddings_gcn_list = []
embeddings_hgcn_list = []
embeddings_mlp_list = []
embeddings_gat_list = []

for seed_int in seeds:
    seed = str(seed_int)
    embedding_name_hybo_L = f"{task_name}/{dataset_name.lower()}/embeddings/hybonet-lorentz.yaml/node_embeddings_{dataset_name.lower()}_Lorentz_model_HyboNet_seed_"
    embedding_name_hybo_E = f"{task_name}/{dataset_name.lower()}/embeddings/hybonet-euclidean.yaml/node_embeddings_{dataset_name.lower()}_Euclidean_model_HyboNet_seed_"
    embedding_name_gcn = f"{task_name}/{dataset_name.lower()}/embeddings/gcn.yaml/node_embeddings_{dataset_name.lower()}_Euclidean_model_GCN_seed_"
    embedding_name_hgcn = f"{task_name}/{dataset_name.lower()}/embeddings/hgcn.yaml/node_embeddings_{dataset_name.lower()}_PoincareBall_model_HGCN_seed_"
    embedding_name_mlp = f"{task_name}/{dataset_name.lower()}/embeddings/mlp.yaml/node_embeddings_{dataset_name.lower()}_Euclidean_model_MLP_seed_"
    embedding_name_gat = f"{task_name}/{dataset_name.lower()}/embeddings/gat.yaml/node_embeddings_{dataset_name.lower()}_Euclidean_model_GAT_seed_"
    embeddings_hybo_L = np.load(embeddings_path + embedding_name_hybo_L + seed + ".npy")[layer[0]]
    embeddings_hybo_E = np.load(embeddings_path + embedding_name_hybo_E + seed + ".npy")[layer[1]]
    embeddings_gcn = np.load(embeddings_path + embedding_name_gcn + seed + ".npy")[layer[2]]
    embeddings_hgcn = np.load(embeddings_path + embedding_name_hgcn + seed + ".npy")[layer[2]]
    embeddings_mlp = np.load(embeddings_path + embedding_name_mlp + seed + ".npy")[layer[3]]
    embeddings_gat = np.load(embeddings_path + embedding_name_gat + seed + ".npy")[layer[4]]
    embeddings_hybo_L_list.append(torch.tensor(embeddings_hybo_L, dtype=torch.float))
    embeddings_hybo_E_list.append(torch.tensor(embeddings_hybo_E, dtype=torch.float))
    embeddings_gcn_list.append(torch.tensor(embeddings_gcn, dtype=torch.float))
    embeddings_hgcn_list.append(torch.tensor(embeddings_hgcn, dtype=torch.float))
    embeddings_mlp_list.append(torch.tensor(embeddings_mlp, dtype=torch.float))
    embeddings_gat_list.append(torch.tensor(embeddings_gat, dtype=torch.float))
features_torch = [torch.tensor(features, dtype=torch.float)]
model_embedding_list = [features_torch, embeddings_mlp_list, embeddings_gcn_list, embeddings_gat_list, embeddings_hybo_E_list, embeddings_hybo_L_list, embeddings_hgcn_list]
embeddings_names = ["Features", "MLP", "GCN", "GAT", "Hybonet Euclidean", "Hybonet Lorentz", "HGCN"]
manifolds = [Euclidean(), Euclidean(), Euclidean(), Euclidean(), Euclidean(), Lorentz(), PoincareBall()]

In [11]:
# fit the linear scaler and compute stress loss for each model and each embedding
model_loss = []
for model_index, embedding_list in enumerate(tqdm(model_embedding_list)):
    embedding_loss = []
    for embedding_index, embedding in enumerate(embedding_list):
        model = DistanceScaler(manifolds[model_index])
        # Loss + optimizer
        criterion = StressLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.05)

        # === Training loop ===
        batch_size = 200000
        num_epochs = 20

        for epoch in range(num_epochs):
            perm = torch.randperm(idx.size(0))
            total_loss = 0.0

            for i in (range(0, idx.size(0), batch_size)):
                batch_idx = idx[perm[i:i+batch_size],:].transpose(0, 1)
                batch_targets = targets[perm[i:i+batch_size]]

                preds = model.decode(embedding, batch_idx)
                loss = criterion(preds, batch_targets)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * batch_idx.size(1)

            avg_loss = total_loss / idx.size(0)


        embedding_loss.append(avg_loss)
    model_loss.append(embedding_loss)
    if model_index==0:
        normalization_factor = np.array(embedding_loss).mean()
        print(f"{embeddings_names[model_index]} \n Stress Loss: {np.array(embedding_loss).mean():.4f}±{np.array(embedding_loss).std():.4f}")
    else:
        print(f"{embeddings_names[model_index]} \n Stress Loss: {np.array(embedding_loss).mean():.4f}±{np.array(embedding_loss).std():.4f} \n Normalized Stress Loss: {np.array(embedding_loss).mean()/normalization_factor:.2f}±{np.array(embedding_loss).std()/normalization_factor:.2f}")

  0%|          | 0/7 [00:00<?, ?it/s]

Features 
 Stress Loss: 0.0842±0.0000
MLP 
 Stress Loss: 0.0573±0.0003 
 Normalized Stress Loss: 0.68±0.00
GCN 
 Stress Loss: 0.0457±0.0011 
 Normalized Stress Loss: 0.54±0.01
GAT 
 Stress Loss: 0.0598±0.0007 
 Normalized Stress Loss: 0.71±0.01
Hybonet Euclidean 
 Stress Loss: 0.0506±0.0042 
 Normalized Stress Loss: 0.60±0.05
Hybonet Lorentz 
 Stress Loss: 0.0316±0.0022 
 Normalized Stress Loss: 0.37±0.03
HGCN 
 Stress Loss: 0.0497±0.0032 
 Normalized Stress Loss: 0.59±0.04
